## Data Cleaning
Tujuan: Membersihkan dan menyiapkan data sebelum digunakan.

## 1. Loading

Pertama, kita perlu mengambil dan melihat data.

In [28]:
import pandas as pd

In [29]:
results = pd.read_csv("../data/raw/results.csv")
former_names = pd.read_csv("../data/raw/former_names.csv")
goalscorers = pd.read_csv("../data/raw/goalscorers.csv")
shootouts = pd.read_csv("../data/raw/shootouts.csv")

In [30]:
tabels_dict= {
    "Results Table": results, 
    "Goalscorers Table": goalscorers, 
    "Former Names Table": former_names, 
    "Shootouts Table": shootouts
}

for nama, tabel in tabels_dict.items():
    print(f"\n{nama}")
    print(tabel.shape)


Results Table
(49477, 9)

Goalscorers Table
(47690, 8)

Former Names Table
(36, 4)

Shootouts Table
(678, 5)


Kita akan gunakan shape untuk membandingkan pada bagian validasi nanti.

## 2. Konversi Tipe Data

Rencana pertama kita adalah mengonversi tipe data agar sesuai dan mudah diproses nantinya. Beberapa kolom yang akan diubah tipe datanya adalah:
- `date` pada kolom `results`
- `start_date` dan `end_date` pada kolom `former_names`
- `date` pada kolom `goalscorers`
- `date` pada kolom `shootouts`

In [31]:
# kolom results
results["date"] = pd.to_datetime(results["date"])

# kolom former_names
former_names["start_date"] = pd.to_datetime(former_names["start_date"])
former_names["end_date"] = pd.to_datetime(former_names["end_date"])

# kolom goalscorers
goalscorers["date"] = pd.to_datetime(goalscorers["date"])

# kolom shootouts
shootouts["date"] = pd.to_datetime(shootouts["date"])


### 2.1 Insight:
Beberapa kolom tanggal sudah berhasil dikonversi dari string menjadi format datetime.

## 3. Implementasi Cleaning Decision Log
Kita akan mengimplementasikan beberapa rencana yang sudah disiapkan pada bagian sebelumnya setelah melakukan berbagai pertimbangan berdasarkan investigasi.

### 3.1 Tabel `results`

Catatan: Hapus 44 pertandingan merupakan jadwal FIFA World Cup 2026 yang belum dimainkan (per-tanggal pengambilan dataset) yang menyebabkan skor belum tersedia.

In [32]:
# hapus baris jika skor kosong (belum dimainkan)
results = results.dropna(subset=["home_score", "away_score"])

### 3.2 Tabel `goalscorers`

Catatan: Terjadi pada beberapa pertandingan Oceania Nations Cup 1980 dengan hasil pertandingan yang tersedia, namun informasi pencetak gol tidak terdokumentasi. Kita memutuskan untuk mempertahankan data karena tidak mempengaruhi data utama (`results`) yang digunakan dalam prediksi. Selain itu, investigasi menemukan kasus injury time (misalnya 45+1' dan 45+2') yang sama-sama dicatat sebagai menit 45 sehingga menghasilkan baris identik sehingga dianggap duplikat.

### 3.3 Tabel `shootouts`

Catatan: Menghapus kolom `first_shooter` karena persentase data kosongnya terlalu banyak dan fitur ini juga tidak relevan dengan tujuan.

In [33]:
shootouts = shootouts.drop(columns=["first_shooter"])

In [34]:
shootouts.head()

,date,home_team,away_team,winner
0,1967-08-22,India,Taiwan,Taiwan
1,1971-11-14,South Korea,Vietnam Republic,South Korea
2,1972-05-07,South Korea,Iraq,Iraq
3,1972-05-17,Thailand,South Korea,South Korea
4,1972-05-19,Thailand,Cambodia,Thailand


### 3.4 Tabel `former_names`

Catatan: Tidak ada tindakan yang dapat dilakukan saat ini pada tabel ini. Perubahann dan penggabungan nama negara dengan hasil pertandingan akan dilakukan pada implementasi berikutnya.

### 3.5 Insight

Secara keseluruhan pada Bagian 3 (Implementasi Cleaning Decision Log), kita telah mengeksekusi rencana pembersihan data dengan rincian sebagai berikut:

- **Tabel `results`**: Menghapus baris data pertandingan masa depan yang belum terlaksana (seperti jadwal FIFA World Cup 2026) karena belum memiliki skor target (`home_score` dan `away_score`).
- **Tabel `goalscorers`**: Mempertahankan seluruh data tanpa ada penghapusan. *Missing values* pada pencetak gol maupun menit, serta baris duplikat akibat pencatatan *injury time*, dibiarkan apa adanya untuk menjaga keutuhan konteks historis.
- **Tabel `shootouts`**: Menghapus kolom `first_shooter` dari *dataframe* karena didominasi oleh data kosong (*NaN*) dan tidak memberikan nilai tambah pada tujuan utama pemodelan prediksi.
- **Tabel `former_names`**: Tidak ada manipulasi atau pembersihan yang dilakukan. Tabel dipertahankan utuh dan baru akan dimanipulasi pada tahap integrasi data berikutnya untuk menangani perubahan nama negara.

Dataset kini telah terbebas dari kolom yang tidak relevan serta anomali data masa depan, sehingga lebih solid untuk dibawa ke tahap selanjutnya.

## 4. Validasi
Validasi data sebelum disimpan. Pertama, validasi data kosong.

In [35]:
tabels_dict = {
    "Results Table": results, 
    "Goalscorers Table": goalscorers, 
    "Former Names Table": former_names, 
    "Shootouts Table": shootouts
}

print("=== ANALISIS DATA KOSONG ===\n")
for nama, tabel in tabels_dict.items():
    print(f"{nama}")
    print("-" * 30)
    print(tabel.isna().sum())
    print("-" * 30)


=== ANALISIS DATA KOSONG ===

Results Table
------------------------------
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64
------------------------------
Goalscorers Table
------------------------------
date           0
home_team      0
away_team      0
team           0
scorer        48
minute       256
own_goal       0
penalty        0
dtype: int64
------------------------------
Former Names Table
------------------------------
current       0
former        0
start_date    0
end_date      0
dtype: int64
------------------------------
Shootouts Table
------------------------------
date         0
home_team    0
away_team    0
winner       0
dtype: int64
------------------------------


Catatan: Data kosong pada tabel `goalscorers` terjadi pada pertandingan grup turnamen Oceania Nations Cup 1980. Karena hasil pertandingan sudah tercatat di tabel utama (`results`), maka data kosong pada kolom `scorer` dan `minutes` kita pertahankan karena tidak akan mempengaruhi data utama.

Selanjutnya kita validasi data duplikat.

In [36]:
print("=== ANALISIS DATA DUPLIKAT ===\n")
for nama, tabel in tabels_dict.items():
    print(f"{nama}")
    print("-" * 30)
    print(tabel.duplicated().sum())
    print("-" * 30)

=== ANALISIS DATA DUPLIKAT ===

Results Table
------------------------------
0
------------------------------
Goalscorers Table
------------------------------
82
------------------------------
Former Names Table
------------------------------
0
------------------------------
Shootouts Table
------------------------------
0
------------------------------


Catatan: Data duplikat pada tabel `goalscorers` adalah kasus injury time (misalnya 45+1' dan 45+2') yang sama-sama dicatat sebagai menit 45 sehingga menghasilkan baris identik. Sehingga kita pertahankan.

In [37]:
for nama, tabel in tabels_dict.items():
    print(f"\n{nama}")
    print(tabel.dtypes)


Results Table
date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

Goalscorers Table
date         datetime64[us]
home_team               str
away_team               str
team                    str
scorer                  str
minute              float64
own_goal               bool
penalty                bool
dtype: object

Former Names Table
current                  str
former                   str
start_date    datetime64[us]
end_date      datetime64[us]
dtype: object

Shootouts Table
date         datetime64[us]
home_team               str
away_team               str
winner                  str
dtype: object


Catatan: Semua tipe data sudah sesuai.

**SHAPE SEBELUM IMPLEMENTASI CLEANING**

Results Table
(49477, 9)

Goalscorers Table
(47690, 8)

Former Names Table
(36, 4)

Shootouts Table
(678, 5)

In [38]:
print("SHAPE SETELAH IMPLEMENTASI CLEANING")

for nama, tabel in tabels_dict.items():    
    print(f"\n{nama}")
    print(tabel.shape)

SHAPE SETELAH IMPLEMENTASI CLEANING

Results Table
(49433, 9)

Goalscorers Table
(47690, 8)

Former Names Table
(36, 4)

Shootouts Table
(678, 4)


Jika dibandingkan pada Bagian 1 (Loading), tervalidasi bahwa ada perubahan akibat penghapusan baris dan kolom.

## 5. Save
Simpan hasil pembersihan.

In [39]:
results.to_csv(
    "../data/interim/results_clean.csv",
    index=False
)

former_names.to_csv(
    "../data/interim/former_names_clean.csv",
    index=False
)

goalscorers.to_csv(
    "../data/interim/goalscorers_clean.csv",
    index=False
)

shootouts.to_csv(
    "../data/interim/shootouts_clean.csv",
    index=False
)

## 6. Final Insight

Proses pada *Data Preprocessing* telah berhasil dieksekusi dengan langkah-langkah yang terstruktur. Berikut adalah rangkuman dari keseluruhan proses persiapan data yang telah dilakukan pada *notebook* ini:

1. **Konversi Tipe Data**: Seluruh kolom yang merepresentasikan waktu (`date` pada `results`, `goalscorers`, `shootouts`; serta `start_date` dan `end_date` pada `former_names`) telah sukses dikonversi dari *string* menjadi tipe `datetime`. Hal ini krusial untuk mempermudah analisis *time-series* atau ekstraksi fitur waktu ke depannya.
2. **Pembersihan Berbasis Investigasi**:
   - **Tabel `results`**: Menghapus 44 baris jadwal pertandingan masa depan yang skornya masih kosong agar target prediksi valid.
   - **Tabel `shootouts`**: Melakukan *drop* pada kolom `first_shooter` yang tidak relevan dan didominasi oleh *missing values*.
   - **Tabel `goalscorers` & `former_names`**: Mempertahankan integritas data asli. Anomali historis, seperti pencatatan *injury time* yang menghasilkan duplikasi semu pada menit ke-45 dan absennya data pencetak gol pada turnamen klasik, dibiarkan apa adanya agar tidak menghilangkan konteks historis.
3. **Validasi Mutu Data**: Pengecekan ulang terhadap *missing values*, data duplikat, dan dimensi (*shape*) *dataframe* membuktikan bahwa pembersihan berjalan persis sesuai rencana awal. Perubahan jumlah baris dan kolom tervalidasi dengan benar.
4. **Penyimpanan Interim**: Keempat *dataframe* yang sudah bersih kini tersimpan dengan aman di direktori `../data/interim/`. Pemisahan folder `raw` dan `interim` ini sangat penting untuk menjaga integritas data mentah.

**Langkah Selanjutnya**: 
Dataset kini memiliki struktur yang jauh lebih solid. Tahap berikutnya adalah memuat data *interim* ini untuk proses *Data Integration* (seperti menangani perubahan nama negara) atau langsung masuk ke fase *Exploratory Data Analysis* (EDA) guna menggali pola-pola menarik sebelum membangun model.